# Piper Trainer RU — Qwen3-ASR + Google Drive

Один Colab для подготовки русского датасета и fine-tune Piper.

Что делает:
- монтирует Google Drive;
- использует `Qwen/Qwen3-ASR-1.7B-hf` для автоматической русской расшифровки;
- режет длинные записи по паузам, приводит их к mono 22050 Гц;
- создаёт `metadata.csv`;
- fine-tune Piper от `ru_RU-dmitri-medium` checkpoint;
- сохраняет checkpoints на Google Drive с настраиваемой частотой и лимитом;
- экспортирует `.onnx + .onnx.json` и ZIP для macOS Piper Voice.

> Для голоса реального человека используйте записи, которые вы вправе использовать. Синтетические записи публичных лиц лучше явно обозначать как синтетические.


In [ ]:
#@title 1. Установка зависимостей
import os, sys, subprocess, textwrap, pathlib

def run(cmd):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)

run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg espeak-ng build-essential cmake ninja-build git")
run("python -m pip install -q -U pip wheel 'setuptools<82' 'jedi>=0.16'")
run("python -m pip install -q 'transformers>=5.13.1' accelerate gradio pydub soundfile librosa huggingface_hub sentencepiece safetensors scikit-build onnx onnxscript")

PIPER_DIR = "/content/piper1-gpl"
PIPER_COMMIT = "5b355b110aecf3de8f4e000ede1ce06831acff35"
if not os.path.exists(PIPER_DIR):
    run("git clone -q https://github.com/OHF-Voice/piper1-gpl.git " + PIPER_DIR)

run(f"cd '{PIPER_DIR}' && git fetch -q origin {PIPER_COMMIT} && git checkout -q {PIPER_COMMIT}")

# Compatibility patch for PyTorch 2.9+ / 2.11:
# Piper's exporter predates the new torch.export-based ONNX path.
_export_py = pathlib.Path(PIPER_DIR) / "src/piper/train/export_onnx.py"
_export_src = _export_py.read_text(encoding="utf-8")
if "dynamo=False" not in _export_src:
    _export_src = _export_src.replace(
        '        dynamic_axes={\n',
        '        dynamo=False,\n        dynamic_axes={\n',
        1,
    )
    _export_py.write_text(_export_src, encoding="utf-8")

run(f"python -m pip install -q -e '{PIPER_DIR}[train]'")
run(f"cd '{PIPER_DIR}' && ./build_monotonic_align.sh")
run(f"cd '{PIPER_DIR}' && python setup.py build_ext --inplace -q")
run(f"cd '{PIPER_DIR}' && python -m piper.train fit --help >/tmp/piper_train_help.txt")
run(f"cd '{PIPER_DIR}' && python -m piper.train.export_onnx --help >/tmp/piper_export_help.txt")

print("Piper training CLI: OK")
print("Piper ONNX export CLI: OK")
print("Piper commit:", PIPER_COMMIT)
print("Готово. Следующая ячейка подключит Google Drive и скачает базовый checkpoint.")


In [ ]:
#@title 2. Google Drive и базовый checkpoint
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from huggingface_hub import hf_hub_download
import os, shutil

ROOT = Path("/content/piper_trainer")
ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = Path("/content/drive/MyDrive/PiperTrainer")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

BASE_DIR = ROOT / "base"
BASE_DIR.mkdir(exist_ok=True)

DRIVE_BASE = DRIVE_ROOT / "_base"
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

def cached_base_file(filename):
    drive_file = DRIVE_BASE / filename
    if not drive_file.exists():
        downloaded = hf_hub_download(
            repo_id="rhasspy/piper-checkpoints",
            repo_type="dataset",
            filename="ru/ru_RU/dmitri/medium/" + filename,
            local_dir=str(DRIVE_BASE / "_download"),
        )
        shutil.copy2(downloaded, drive_file)

    local_file = BASE_DIR / filename
    if (
        not local_file.exists()
        or local_file.stat().st_size != drive_file.stat().st_size
    ):
        shutil.copy2(drive_file, local_file)
    return str(local_file)

DMITRI_CKPT = cached_base_file("epoch=5589-step=1478840.ckpt")
DMITRI_CONFIG = cached_base_file("config.json")

print("Базовый checkpoint:", DMITRI_CKPT)
print("Постоянный кэш базы:", DRIVE_BASE)
print("Google Drive:", DRIVE_ROOT)


In [ ]:
#@title 3. Backend: подготовка датасета, ASR и хранение
import os, gc, csv, json, math, shutil, zipfile, subprocess, threading, time, re
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from transformers import AutoProcessor, AutoModelForMultimodalLM

WORK_ROOT = Path("/content/piper_trainer")
ASR_MODEL = "Qwen/Qwen3-ASR-1.7B-hf"
_asr = None
_train_process = None

def environment_status():
    parts = []
    parts.append(f"Python: {sys.version.split()[0]}")
    parts.append(f"PyTorch: {torch.__version__}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram = props.total_memory / (1024 ** 3)
        parts.append(f"GPU: {torch.cuda.get_device_name(0)}")
        parts.append(f"VRAM: {vram:.1f} ГБ")
        parts.append("CUDA: доступна")
    else:
        parts.append("GPU: НЕ НАЙДЕН")
        parts.append("Откройте Runtime → Change runtime type → T4 GPU")
    parts.append(
        "Google Drive: подключён"
        if DRIVE_ROOT.exists()
        else "Google Drive: не подключён"
    )
    parts.append(f"Qwen ASR: {ASR_MODEL}")
    parts.append(f"Базовый Piper checkpoint: {Path(DMITRI_CKPT).name}")
    return "\n".join(parts)

def safe_name(name: str) -> str:
    name = re.sub(r"[^0-9A-Za-zА-Яа-яЁё_-]+", "_", name.strip())
    return name.strip("_") or "voice"

def project_paths(project: str):
    project = safe_name(project)
    local = WORK_ROOT / project
    drive_p = DRIVE_ROOT / project
    audio = local / "dataset" / "wav"
    audio.mkdir(parents=True, exist_ok=True)
    drive_p.mkdir(parents=True, exist_ok=True)
    return local, drive_p, audio

def load_asr():
    global _asr
    if _asr is None:
        if not torch.cuda.is_available():
            raise RuntimeError("GPU не найден. В Colab выберите Runtime → Change runtime type → T4 GPU.")
        print("Загружаю", ASR_MODEL)
        processor = AutoProcessor.from_pretrained(ASR_MODEL)
        model = AutoModelForMultimodalLM.from_pretrained(
            ASR_MODEL,
            dtype=torch.float16,
            device_map="auto",
        )
        model.eval()
        _asr = (processor, model)
    return _asr

def transcribe_file(path):
    processor, model = load_asr()
    inputs = processor.apply_transcription_request(
        audio=str(path),
        language="ru",
    ).to(model.device, model.dtype)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=512)

    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    text = processor.decode(
        generated_ids,
        return_format="transcription_only",
    )
    if isinstance(text, (list, tuple)):
        if len(text) == 1:
            text = text[0]
        else:
            text = " ".join(str(part) for part in text)
    return normalize_text(text)

def test_qwen_asr(file_path):
    if not file_path:
        raise ValueError("Выберите один аудиофайл для проверки Qwen.")
    text = transcribe_file(file_path)
    return "Qwen3-ASR распознал:\n" + text

def unload_asr():
    global _asr
    _asr = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def normalize_clip_loudness(clip, target_dbfs=-20.0, peak_ceiling_dbfs=-1.0):
    """Normalize one speech clip while keeping at least 1 dB of peak headroom."""
    if len(clip) == 0 or clip.rms == 0:
        return clip
    gain = float(target_dbfs) - float(clip.dBFS)
    clip = clip.apply_gain(gain)
    if clip.max_dBFS > float(peak_ceiling_dbfs):
        clip = clip.apply_gain(float(peak_ceiling_dbfs) - float(clip.max_dBFS))
    return clip

def _quietest_cut_ms(audio, start_ms, end_ms, min_piece_ms, max_piece_ms):
    """Choose a low-energy cut instead of blindly cutting at max_piece_ms."""
    lo = start_ms + max(int(min_piece_ms), int(max_piece_ms * 0.55))
    hi = min(start_ms + int(max_piece_ms), end_ms - int(min_piece_ms))
    if hi <= lo:
        return min(start_ms + int(max_piece_ms), end_ms)

    target = start_ms + int(max_piece_ms * 0.82)
    best = None
    # 80 ms window every 20 ms is enough to find inter-word valleys.
    for cut in range(lo, hi + 1, 20):
        window = audio[max(start_ms, cut - 40):min(end_ms, cut + 40)]
        db = -100.0 if len(window) == 0 or window.rms == 0 else float(window.dBFS)
        distance_penalty = abs(cut - target) / max_piece_ms * 4.0
        score = db + distance_penalty
        if best is None or score < best[0]:
            best = (score, cut)
    return best[1] if best else min(start_ms + int(max_piece_ms), end_ms)

def _split_long_nonsilent_range(audio, start_ms, end_ms, min_piece_ms, max_piece_ms):
    """Split continuous speech only at the quietest available point."""
    out = []
    cur = int(start_ms)
    end_ms = int(end_ms)
    while end_ms - cur > int(max_piece_ms):
        cut = _quietest_cut_ms(
            audio,
            cur,
            end_ms,
            min_piece_ms=min_piece_ms,
            max_piece_ms=max_piece_ms,
        )
        if cut <= cur:
            cut = min(cur + int(max_piece_ms), end_ms)
        out.append((cur, cut))
        cur = cut
    if end_ms > cur:
        out.append((cur, end_ms))
    return out

def smart_segment_ranges(
    audio,
    min_silence_ms=250,
    silence_db=-35.0,
    min_sec=1.5,
    max_sec=9.0,
    padding_ms=350,
    merge_gap_ms=650,
):
    """Create TTS-friendly phrases using real pauses instead of fixed-time chopping."""
    min_ms = int(float(min_sec) * 1000)
    max_ms = int(float(max_sec) * 1000)

    raw = detect_nonsilent(
        audio,
        min_silence_len=int(min_silence_ms),
        silence_thresh=float(silence_db),
        seek_step=10,
    )
    if not raw:
        return []

    # Pack neighboring speech islands only while the whole phrase stays under max_ms.
    packed = []
    current = None
    for start, end in raw:
        start, end = int(start), int(end)
        if end - start > max_ms:
            if current is not None:
                packed.append(tuple(current))
                current = None
            packed.extend(
                _split_long_nonsilent_range(
                    audio,
                    start,
                    end,
                    min_piece_ms=min_ms,
                    max_piece_ms=max_ms,
                )
            )
            continue

        if current is None:
            current = [start, end]
            continue

        gap = start - current[1]
        if gap <= int(merge_gap_ms) and end - current[0] <= max_ms:
            current[1] = end
        else:
            packed.append(tuple(current))
            current = [start, end]

    if current is not None:
        packed.append(tuple(current))

    # Absorb very short phrases into a neighbor when that still respects max_ms.
    packed = [list(x) for x in packed]
    i = 0
    while i < len(packed):
        duration = packed[i][1] - packed[i][0]
        if duration < min_ms and len(packed) > 1:
            if (
                i + 1 < len(packed)
                and packed[i + 1][0] - packed[i][1] <= int(merge_gap_ms)
                and packed[i + 1][1] - packed[i][0] <= max_ms
            ):
                packed[i + 1][0] = packed[i][0]
                packed.pop(i)
                continue
            if (
                i > 0
                and packed[i][0] - packed[i - 1][1] <= int(merge_gap_ms)
                and packed[i][1] - packed[i - 1][0] <= max_ms
            ):
                packed[i - 1][1] = packed[i][1]
                packed.pop(i)
                i -= 1
                continue
        i += 1

    final = []
    for start, end in packed:
        start = max(0, int(start) - int(padding_ms))
        end = min(len(audio), int(end) + int(padding_ms))
        if end - start >= min_ms:
            final.append((start, end))
    return final


def phrase_quality_reason(text, duration):
    text = normalize_text(text)
    letters = len(re.findall(r"[A-Za-zА-Яа-яЁё]", text))
    cps = len(text) / max(float(duration), 0.001)
    if not text:
        return "пустая расшифровка"
    if letters < 8:
        return "слишком короткая фраза"
    if cps < 5.0:
        return f"слишком мало текста для {duration:.1f} сек"
    if cps > 21.0:
        return f"слишком много текста для {duration:.1f} сек"
    if text[-1] not in ".!?":
        return "фраза оборвана: нет конечного знака"
    low = text.lower()
    if re.search(r"\b(и и и|это это это|что что что|ну ну ну)\b", low):
        return "подозрительное повторение"
    if text.startswith(("['", '[\"')) or text.endswith(("']", '\"]')):
        return "служебные скобки ASR"
    return None

def normalize_text(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = text.replace("…", "...")
    return text


In [ ]:
AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".mp4"}

def expand_input_files(files, local):
    if not isinstance(files, (list, tuple)):
        files = [files]
    imported = local / "_imports"
    if imported.exists():
        shutil.rmtree(imported)
    imported.mkdir(parents=True, exist_ok=True)

    out = []
    serial = 0
    for file_obj in files:
        src = Path(file_obj)
        if src.suffix.lower() != ".zip":
            if src.suffix.lower() in AUDIO_EXTS:
                out.append(src)
            continue

        with zipfile.ZipFile(src) as z:
            for info in z.infolist():
                if info.is_dir():
                    continue
                suffix = Path(info.filename).suffix.lower()
                if suffix not in AUDIO_EXTS:
                    continue
                serial += 1
                target = imported / f"zip_{serial:05d}{suffix}"
                with z.open(info) as source, target.open("wb") as dest:
                    shutil.copyfileobj(source, dest)
                out.append(target)
    return out

def prepare_dataset(files, project, min_silence_ms, silence_db, min_sec, max_sec, padding_ms, status=None):
    if not files:
        raise ValueError("Сначала загрузите хотя бы один аудиофайл.")
    project = safe_name(project)
    local, drive_p, wav_dir = project_paths(project)

    # очищаем только текущий локальный датасет
    dataset_dir = local / "dataset"
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)
    wav_dir = dataset_dir / "wav"
    wav_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    rejected = []
    counter = 0

    files = expand_input_files(files, local)
    if not files:
        raise ValueError("В загрузке не найдено поддерживаемых аудиофайлов.")

    try:
        load_asr()
        for file_obj in files:
            src = Path(file_obj)
            audio = AudioSegment.from_file(src).set_channels(1).set_frame_rate(22050)
            expanded = smart_segment_ranges(
                audio,
                min_silence_ms=min_silence_ms,
                silence_db=silence_db,
                min_sec=min_sec,
                max_sec=max_sec,
                padding_ms=padding_ms,
            )

            for start, end in expanded:
                counter += 1
                name = f"{project}_{counter:06d}.wav"
                out = wav_dir / name
                clip = normalize_clip_loudness(audio[start:end])
                clip.export(out, format="wav", parameters=["-acodec", "pcm_s16le"])

                text = transcribe_file(out)
                duration = len(clip) / 1000.0
                chars_per_sec = len(text.strip()) / max(duration, 0.001)
                reason = phrase_quality_reason(text, duration)

                if reason:
                    rejected.append({
                        "file": name,
                        "reason": reason,
                        "duration": round(duration, 3),
                        "text": text,
                    })
                    out.unlink(missing_ok=True)
                    continue

                rows.append({
                    "file": name,
                    "text": text,
                    "duration": round(duration, 3),
                    "source": src.name,
                    "start_sec": round(start / 1000.0, 3),
                    "end_sec": round(end / 1000.0, 3),
                    "chars_per_sec": round(chars_per_sec, 3),
                    "rms_dbfs": round(float(clip.dBFS), 3),
                    "peak_dbfs": round(float(clip.max_dBFS), 3),
                })
    finally:
        unload_asr()

    if not rows:
        raise RuntimeError("Не получилось получить ни одной пригодной фразы. Попробуйте снизить порог тишины.")

    df = pd.DataFrame(rows)
    review_csv = dataset_dir / "review.csv"
    df.to_csv(review_csv, index=False, encoding="utf-8")

    metadata = dataset_dir / "metadata.csv"
    with metadata.open("w", encoding="utf-8", newline="") as f:
        for row in rows:
            f.write(f"{row['file']}|{row['text']}\n")

    report = {
        "phrases": int(len(rows)),
        "rejected": int(len(rejected)),
        "minutes": round(float(df["duration"].sum()) / 60.0, 3),
        "duration_min": round(float(df["duration"].min()), 3),
        "duration_median": round(float(df["duration"].median()), 3),
        "duration_max": round(float(df["duration"].max()), 3),
        "rms_dbfs_min": round(float(df["rms_dbfs"].min()), 3),
        "rms_dbfs_median": round(float(df["rms_dbfs"].median()), 3),
        "rms_dbfs_max": round(float(df["rms_dbfs"].max()), 3),
        "longer_than_10_sec": int((df["duration"] > 10).sum()),
        "shorter_than_2_sec": int((df["duration"] < 2).sum()),
        "settings": {
            "min_silence_ms": int(min_silence_ms),
            "silence_db": float(silence_db),
            "min_sec": float(min_sec),
            "max_sec": float(max_sec),
            "padding_ms": int(padding_ms),
            "target_dbfs": -20.0,
            "peak_ceiling_dbfs": -1.0,
        },
    }
    (dataset_dir / "dataset_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    if rejected:
        pd.DataFrame(rejected).to_csv(
            dataset_dir / "rejected.csv",
            index=False,
            encoding="utf-8",
        )

    # сохраняем подготовленный датасет на Drive
    drive_dataset = drive_p / "dataset"
    if drive_dataset.exists():
        shutil.rmtree(drive_dataset)
    shutil.copytree(dataset_dir, drive_dataset)

    summary = (
        f"Готово: {len(rows)} фраз, "
        f"{df['duration'].sum()/60:.1f} минут. "
        f"Отклонено: {len(rejected)}. "
        f"Длительность: {df['duration'].min():.1f}–{df['duration'].max():.1f} сек, "
        f"медиана {df['duration'].median():.1f} сек. "
        f"Громкость нормализована около -20 dBFS. "
        f"Датасет: {drive_dataset}"
    )
    editor_text = "\n".join(f"{row['file']}|{row['text']}" for row in rows)
    return df, summary, str(review_csv), editor_text

def apply_text_review(project, editor_text):
    project = safe_name(project)
    local, drive_p, _ = project_paths(project)
    dataset_dir = local / "dataset"
    wav_dir = dataset_dir / "wav"

    clean_rows = []
    errors = []
    for line_no, raw in enumerate(str(editor_text).splitlines(), start=1):
        raw = raw.strip()
        if not raw:
            continue
        if "|" not in raw:
            errors.append(f"строка {line_no}: нет символа |")
            continue
        filename, text = raw.split("|", 1)
        filename = filename.strip()
        text = normalize_text(text)
        if not filename or not text:
            errors.append(f"строка {line_no}: пустое имя файла или текст")
            continue
        if Path(filename).name != filename or not (wav_dir / filename).exists():
            errors.append(f"строка {line_no}: аудиофайл {filename} не найден")
            continue
        clean_rows.append((filename, text))

    if errors:
        raise ValueError("Исправьте ошибки:\n" + "\n".join(errors[:20]))
    if not clean_rows:
        raise ValueError("Нет ни одной строки вида имя.wav|текст")

    metadata = dataset_dir / "metadata.csv"
    with metadata.open("w", encoding="utf-8", newline="") as f:
        for filename, text in clean_rows:
            f.write(f"{filename}|{text}\n")

    drive_dataset = drive_p / "dataset"
    drive_dataset.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metadata, drive_dataset / "metadata.csv")
    return (
        f"Сохранено {len(clean_rows)} фраз. "
        f"metadata.csv обновлён и скопирован на Google Drive."
    )

def apply_review(project, review_file):
    project = safe_name(project)
    local, drive_p, wav_dir = project_paths(project)
    dataset_dir = local / "dataset"
    review_csv = Path(review_file) if review_file else dataset_dir / "review.csv"
    df = pd.read_csv(review_csv)

    metadata = dataset_dir / "metadata.csv"
    with metadata.open("w", encoding="utf-8", newline="") as f:
        for _, row in df.iterrows():
            text = normalize_text(row["text"])
            if text:
                f.write(f"{row['file']}|{text}\n")

    drive_dataset = drive_p / "dataset"
    drive_dataset.mkdir(parents=True, exist_ok=True)
    shutil.copy2(metadata, drive_dataset / "metadata.csv")
    shutil.copy2(review_csv, drive_dataset / "review.csv")
    return f"Исправления применены. metadata.csv обновлён: {metadata}"


In [ ]:
#@title 4. Backend: обучение, checkpoints и экспорт
def restore_dataset_from_drive(project):
    project = safe_name(project)
    local, drive_p, wav_dir = project_paths(project)
    src = drive_p / "dataset"
    dst = local / "dataset"
    if not src.exists():
        raise FileNotFoundError(f"На Google Drive нет датасета для проекта {project}")
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    return dst

def newest_checkpoint(path: Path):
    files = [p for p in path.glob("*.ckpt") if p.name != "last.ckpt"]
    return max(files, key=lambda p: p.stat().st_mtime) if files else None

def sync_checkpoints(local_ckpt: Path, drive_ckpt: Path, keep_last: int):
    drive_ckpt.mkdir(parents=True, exist_ok=True)
    for src in local_ckpt.glob("*.ckpt"):
        dst = drive_ckpt / src.name
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            try:
                shutil.copy2(src, dst)
            except Exception:
                pass

    if keep_last > 0:
        for folder in (drive_ckpt, local_ckpt):
            ckpts = sorted(
                folder.glob("*.ckpt"),
                key=lambda p: p.stat().st_mtime,
                reverse=True,
            )
            for old in ckpts[int(keep_last):]:
                old.unlink(missing_ok=True)

def checkpoint_epoch(path):
    try:
        data = torch.load(str(path), map_location="cpu", weights_only=False)
        epoch = int(data.get("epoch", -1))
        del data
        gc.collect()
        return epoch
    except Exception as e:
        print("Не удалось прочитать номер эпохи checkpoint:", e)
        return -1


def resolve_start_checkpoint(project, start_mode):
    project = safe_name(project)
    local, drive_p, _ = project_paths(project)

    if start_mode == "Последний checkpoint с Google Drive":
        candidates = []

        local_full = newest_checkpoint(local / "checkpoints")
        if local_full:
            candidates.append((local_full, "resume"))

        drive_full = newest_checkpoint(drive_p / "checkpoints")
        if drive_full:
            candidates.append((drive_full, "resume"))

        drive_recovery = newest_checkpoint(drive_p / "recovery")
        if drive_recovery:
            candidates.append((drive_recovery, "recovery"))

        if candidates:
            path, kind = max(
                candidates,
                key=lambda item: item[0].stat().st_mtime,
            )
            return str(path), kind

    return str(DMITRI_CKPT), "base"


def train_voice(
    project,
    additional_epochs,
    batch_size,
    save_mode,
    save_every,
    keep_last,
    drive_backup_every,
    start_mode,
):
    global _train_process
    project = safe_name(project)
    unload_asr()

    local, drive_p, _ = project_paths(project)
    dataset = local / "dataset"
    if not (dataset / "metadata.csv").exists():
        restore_dataset_from_drive(project)

    wav_dir = dataset / "wav"
    metadata = dataset / "metadata.csv"
    cache_dir = local / "cache"
    config_path = drive_p / f"ru_RU-{project}-medium.onnx.json"
    local_ckpt = local / "checkpoints"
    drive_recovery = drive_p / "recovery"

    cache_dir.mkdir(parents=True, exist_ok=True)
    local_ckpt.mkdir(parents=True, exist_ok=True)
    drive_recovery.mkdir(parents=True, exist_ok=True)

    start_ckpt, start_kind = resolve_start_checkpoint(
        project,
        start_mode,
    )
    is_resume = start_kind == "resume"
    start_epoch = checkpoint_epoch(start_ckpt)
    additional_epochs = max(1, int(additional_epochs))

    target_max_epochs = (
        start_epoch + 1 + additional_epochs
        if is_resume and start_epoch >= 0
        else additional_epochs
    )

    save_every = max(1, int(save_every))
    keep_last = max(1, int(keep_last))
    drive_backup_every = max(1, int(drive_backup_every))

    runner = local / "run_piper_training.py"
    if save_mode == "Каждые N эпох":
        schedule_arg = f"every_n_epochs={save_every}"
    else:
        schedule_arg = f"every_n_train_steps={save_every}"

    runner.write_text(
        f"""
import shutil
from pathlib import Path

import torch
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
import piper.train.__main__ as piper_train


class PruningModelCheckpoint(ModelCheckpoint):
    def _save_checkpoint(self, trainer, filepath):
        super()._save_checkpoint(trainer, filepath)
        root = Path(self.dirpath)
        keep = {keep_last}
        named = sorted(
            root.glob("*.ckpt"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        for old in named[keep:]:
            try:
                old.unlink()
            except FileNotFoundError:
                pass


class DriveRecoveryCallback(Callback):
    def __init__(self, drive_dir, temp_file, interval):
        super().__init__()
        self.drive_dir = Path(drive_dir)
        self.temp_file = Path(temp_file)
        self.interval = max(1, int(interval))
        self.last_saved_epoch = None
        self.drive_dir.mkdir(parents=True, exist_ok=True)

    def _save_recovery(self, trainer, pl_module):
        epoch = int(trainer.current_epoch)
        step = int(trainer.global_step)

        state_dict = {{
            key: value.detach().cpu()
            for key, value in pl_module.state_dict().items()
        }}
        payload = {{
            "state_dict": state_dict,
            "epoch": epoch,
            "global_step": step,
            "piper_recovery_weights_only": True,
        }}

        torch.save(payload, self.temp_file)
        target = self.drive_dir / (
            f"recovery-{{epoch:04d}}-{{step}}.ckpt"
        )
        shutil.copy2(self.temp_file, target)
        self.temp_file.unlink(missing_ok=True)
        self.last_saved_epoch = epoch
        print(
            f"[Drive recovery] saved {{target.name}} "
            f"({{target.stat().st_size / 1024 / 1024:.1f}} MiB)"
        )

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = int(trainer.current_epoch)
        if (epoch + 1) % self.interval == 0:
            self._save_recovery(trainer, pl_module)

    def on_train_end(self, trainer, pl_module):
        epoch = int(trainer.current_epoch)
        if self.last_saved_epoch != epoch:
            self._save_recovery(trainer, pl_module)


checkpoint = PruningModelCheckpoint(
    dirpath={str(local_ckpt)!r},
    save_top_k=-1,
    save_last=False,
    filename={project!r} + "-{{epoch:04d}}-{{step}}",
    auto_insert_metric_name=False,
    {schedule_arg},
)

recovery = DriveRecoveryCallback(
    drive_dir={str(drive_recovery)!r},
    temp_file={str(local / "recovery-tmp.ckpt")!r},
    interval={drive_backup_every},
)

piper_train._DEFAULT_CALLBACKS = [checkpoint, recovery]
piper_train.main()
""".strip() + "\n",
        encoding="utf-8",
    )

    cmd = [
        sys.executable,
        str(runner),
        "fit",
        "--data.voice_name",
        project,
        "--data.csv_path",
        str(metadata),
        "--data.audio_dir",
        str(wav_dir),
        "--model.sample_rate",
        "22050",
        "--data.espeak_voice",
        "ru",
        "--data.cache_dir",
        str(cache_dir),
        "--data.config_path",
        str(config_path),
        "--data.batch_size",
        str(int(batch_size)),
    ]

    if is_resume:
        cmd += ["--ckpt_path", start_ckpt]
    else:
        cmd += ["--model.warmstart_ckpt", start_ckpt]

    cmd += [
        "--trainer.max_epochs",
        str(target_max_epochs),
        "--trainer.accelerator",
        "gpu",
        "--trainer.devices",
        "1",
        "--trainer.precision",
        "16-mixed",
        "--trainer.log_every_n_steps",
        "1",
        "--trainer.enable_checkpointing",
        "true",
    ]

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

    if start_kind == "resume":
        start_label = "точный resume полного checkpoint"
    elif start_kind == "recovery":
        start_label = "warm-start recovery с Drive (без optimizer)"
    else:
        start_label = "warm-start от Dmitri"

    log_file = drive_p / "training.log"
    header = (
        "Старт обучения\n"
        f"Проект: {project}\n"
        f"Checkpoint: {start_ckpt}\n"
        f"Режим старта: {start_label}\n"
        f"Эпоха источника: {start_epoch if start_epoch >= 0 else 'неизвестна'}\n"
        f"Дополнительных эпох: {additional_epochs}\n"
        f"Целевая max_epochs Lightning: {target_max_epochs}\n"
        f"Полные checkpoints локально: {local_ckpt}\n"
        f"Recovery на Drive каждые {drive_backup_every} эпох: {drive_recovery}\n\n"
    )
    yield header

    _train_process = subprocess.Popen(
        cmd,
        cwd="/content/piper1-gpl",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    collected = [header]
    with log_file.open("a", encoding="utf-8") as lf:
        for line in _train_process.stdout:
            collected.append(line)
            lf.write(line)
            lf.flush()
            yield "".join(collected[-220:])

    code = _train_process.wait()
    _train_process = None

    if code == 0:
        yield (
            "".join(collected[-220:])
            + "\n\nОбучение завершено. "
            "Полные checkpoints остались локально, recovery сохранены на Google Drive."
        )
    else:
        yield (
            "".join(collected[-220:])
            + f"\n\nОбучение остановилось с кодом {code}."
        )


def stop_training():
    global _train_process
    if _train_process is not None and _train_process.poll() is None:
        _train_process.terminate()
        return "Запрошена остановка обучения. Текущие сохранённые checkpoints останутся."
    return "Активного обучения сейчас нет."

def export_voice(project):
    project = safe_name(project)
    local, drive_p, _ = project_paths(project)
    drive_ckpt = drive_p / "checkpoints"
    ckpt = newest_checkpoint(local / "checkpoints")
    if ckpt is None:
        ckpt = newest_checkpoint(drive_ckpt)
    if ckpt is None:
        raise FileNotFoundError("Checkpoint не найден.")

    config = local / f"ru_RU-{project}-medium.onnx.json"
    if not config.exists():
        drive_config = drive_p / f"ru_RU-{project}-medium.onnx.json"
        if drive_config.exists():
            shutil.copy2(drive_config, config)
        else:
            raise FileNotFoundError("Не найден config JSON, созданный Piper во время обучения.")

    out_dir = local / "export"
    out_dir.mkdir(parents=True, exist_ok=True)
    model_name = f"ru_RU-{project}-medium"
    onnx = out_dir / f"{model_name}.onnx"
    json_out = out_dir / f"{model_name}.onnx.json"

    export_env = os.environ.copy()
    export_env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    subprocess.run(
        [
            sys.executable, "-m", "piper.train.export_onnx",
            "--checkpoint", str(ckpt),
            "--output-file", str(onnx),
        ],
        cwd="/content/piper1-gpl",
        env=export_env,
        check=True,
    )
    shutil.copy2(config, json_out)

    zip_path = out_dir / f"{model_name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(onnx, onnx.name)
        z.write(json_out, json_out.name)

    drive_export = drive_p / "export"
    drive_export.mkdir(parents=True, exist_ok=True)
    shutil.copy2(onnx, drive_export / onnx.name)
    shutil.copy2(json_out, drive_export / json_out.name)
    shutil.copy2(zip_path, drive_export / zip_path.name)

    return str(zip_path), f"Экспорт готов: {drive_export / zip_path.name}"


In [ ]:
#@title 5. Запуск веб-интерфейса
import gradio as gr

with gr.Blocks(title="Piper Trainer RU") as demo:
    gr.Markdown(
        "# Piper Trainer RU\n"
        "Qwen3-ASR 1.7B → автоматическая нарезка → Piper fine-tune → Google Drive"
    )

    with gr.Row():
        env_btn = gr.Button("Проверить окружение")
        env_status = gr.Textbox(label="Окружение", lines=7)
    env_btn.click(environment_status, outputs=env_status)
    demo.load(environment_status, outputs=env_status)

    with gr.Tab("1. Датасет"):
        project = gr.Textbox(
            label="Название проекта",
            value="my_voice",
            info="Латиница/кириллица допустимы. Пробелы будут заменены на подчёркивания."
        )
        files = gr.Files(
            label="Длинные записи или ZIP с аудиофайлами",
            file_types=["audio", ".zip"],
            type="filepath",
        )

        with gr.Row():
            min_silence = gr.Slider(80, 800, value=120, step=20, label="Пауза для разреза, мс")
            silence_db = gr.Slider(-60, -20, value=-35, step=1, label="Порог тишины, дБ")
        with gr.Row():
            min_sec = gr.Slider(0.5, 5.0, value=1.5, step=0.1, label="Минимум фразы, сек")
            max_sec = gr.Slider(3, 20, value=9, step=1, label="Максимум фразы, сек")
            padding = gr.Slider(0, 500, value=80, step=20, label="Запас по краям, мс")

        with gr.Row():
            asr_test_file = gr.File(
                label="Одно аудио для проверки Qwen",
                file_types=["audio"],
                type="filepath",
            )
            asr_test_btn = gr.Button("Проверить Qwen на одном аудиофайле")
        asr_test_result = gr.Textbox(label="Результат проверки Qwen", lines=4)

        prepare_btn = gr.Button("Распознать и подготовить датасет", variant="primary")
        dataset_status = gr.Textbox(label="Статус", lines=3)
        table = gr.Dataframe(
            headers=["file", "text", "duration", "source", "start_sec", "end_sec"],
            label="Результат. Текст можно исправить прямо в таблице.",
            interactive=True,
        )
        review_file = gr.File(label="review.csv")
        metadata_editor = gr.Textbox(
            label="Текстовый редактор для VoiceOver: имя.wav|текст",
            lines=18,
            info=(
                "Одна фраза на строку. Можно исправлять только текст после первого символа |. "
                "Этот редактор проще таблицы для VoiceOver."
            ),
        )
        with gr.Row():
            apply_text_btn = gr.Button("Сохранить текстовый редактор", variant="primary")
            apply_btn = gr.Button("Применить исправления из таблицы")

        def save_table(project_name, data):
            p = safe_name(project_name)
            local, drive_p, _ = project_paths(p)
            dataset = local / "dataset"
            df = pd.DataFrame(data)
            path = dataset / "review.csv"
            df.to_csv(path, index=False, encoding="utf-8")
            return apply_review(p, str(path))

        asr_test_btn.click(
            test_qwen_asr,
            inputs=asr_test_file,
            outputs=asr_test_result,
        )
        prepare_btn.click(
            prepare_dataset,
            inputs=[files, project, min_silence, silence_db, min_sec, max_sec, padding],
            outputs=[table, dataset_status, review_file, metadata_editor],
        )
        apply_text_btn.click(
            apply_text_review,
            inputs=[project, metadata_editor],
            outputs=dataset_status,
        )
        apply_btn.click(save_table, inputs=[project, table], outputs=dataset_status)

    with gr.Tab("2. Обучение"):
        gr.Markdown("Перед обучением Qwen ASR выгружается из GPU. Полные checkpoints хранятся локально, а компактные recovery-копии периодически сохраняются на Google Drive.")
        with gr.Row():
            epochs = gr.Number(
                value=500,
                precision=0,
                label="Сколько дополнительных эпох обучить",
                info="Считается поверх эпохи выбранного checkpoint. Для первого теста можно поставить 10–20.",
            )
            batch = gr.Dropdown([4, 6, 8, 12, 16], value=8, label="Batch size")
        with gr.Row():
            save_mode = gr.Radio(
                ["Каждые N эпох", "Каждые N шагов"],
                value="Каждые N эпох",
                label="Когда сохранять checkpoint",
            )
            save_every = gr.Number(value=5, precision=0, label="N")
            keep_last = gr.Number(
                value=3,
                precision=0,
                label="Сколько последних полных checkpoints хранить локально",
            )
            drive_backup_every = gr.Number(
                value=100,
                precision=0,
                label="Recovery на Google Drive каждые N эпох",
                info="Recovery весит примерно в 3 раза меньше полного checkpoint и нужен для восстановления после отключения Colab.",
            )
        start_mode = gr.Radio(
            ["Dmitri medium (база)", "Последний checkpoint с Google Drive"],
            value="Dmitri medium (база)",
            label="Откуда продолжать",
        )

        with gr.Row():
            train_btn = gr.Button("Начать обучение", variant="primary")
            stop_btn = gr.Button("Остановить")
        train_log = gr.Textbox(label="Лог обучения", lines=22, autoscroll=True)

        train_btn.click(
            train_voice,
            inputs=[
                project,
                epochs,
                batch,
                save_mode,
                save_every,
                keep_last,
                drive_backup_every,
                start_mode,
            ],
            outputs=train_log,
        )
        stop_btn.click(stop_training, outputs=train_log)

    with gr.Tab("3. Экспорт"):
        export_btn = gr.Button("Экспортировать последний checkpoint в ONNX", variant="primary")
        exported_zip = gr.File(label="ZIP для Piper Voice")
        export_status = gr.Textbox(label="Статус")
        export_btn.click(export_voice, inputs=[project], outputs=[exported_zip, export_status])

    with gr.Tab("Справка"):
        gr.Markdown(
            "### Рекомендуемые настройки для T4\n"
            "- Batch size: 8. Если VRAM заканчивается — 4 или 6.\n"
            "- Для smoke test: 10–20 дополнительных эпох. Для нормального обучения — больше.\n"
            "- Checkpoint: каждые 5–10 эпох.\n"
            "- Хранить: 3 последних, потому что каждый checkpoint большой.\n"
            "- Для первого опыта достаточно 30–60 минут чистой речи, но больше обычно лучше.\n\n"
            "### Google Drive\n"
            "Всё лежит в `MyDrive/PiperTrainer/<проект>/`."
        )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)
